In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

: 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [ ]:
sns.set(style="whitegrid")

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Shikher-jain/Xylopy_AI/refs/heads/main/EmployeeAttrition_ShikherJain/HR_Attrition.csv")

df.head(10)

In [ ]:
print("Shape:", df.shape)
print(df.info())

In [ ]:
df['Attrition'].value_counts()

In [ ]:
attrition_rate = df['Attrition'].value_counts(normalize=True) * 100
print(attrition_rate)

In [ ]:
drop_cols = ['EmployeeNumber', 'Over18', 'StandardHours', 'EmployeeCount']
df.drop(columns=drop_cols, inplace=True)

In [ ]:
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop('Attrition', axis=1)
y = df['Attrition']

cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

print("Categorical:", len(cat_cols))
print("Numerical:", len(num_cols))

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])

In [ ]:
import os

if not os.path.exists('charts'):
    os.makedirs('charts')

In [ ]:
dept_attrition = df.groupby('Department')['Attrition'].mean() * 100
print(dept_attrition)

dept_attrition.plot(kind='bar', title="Attrition Rate by Department")
plt.show()
plt.savefig("charts/Attrition_By_Department.png")

: 

In [ ]:
role_attrition = df.groupby('JobRole')['Attrition'].mean() * 100
role_attrition.sort_values().plot(kind='barh', title="Attrition by Job Role")
plt.show()
plt.savefig("charts/Attrition_By_JobRole.png")

In [ ]:
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df)
plt.title("Income vs Attrition")
plt.show()
plt.savefig("charts/Income_vs_Attrition.png")

In [ ]:
wlb_attrition = df.groupby('WorkLifeBalance')['Attrition'].mean() * 100
wlb_attrition.plot(kind='bar', title="Attrition vs Work-Life Balance")
plt.show()
plt.savefig("charts/Attrition_By_WorkLifeBalance.png")

In [ ]:
sns.histplot(data=df, x='YearsAtCompany', hue='Attrition', bins=20)
plt.title("Tenure vs Attrition")
plt.show()
plt.savefig("charts/Tenure_vs_Attrition.png")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
model_lr = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

In [ ]:
model_rf = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

In [ ]:
model_gb = Pipeline([
    ('prep', preprocessor),
    ('clf', GradientBoostingClassifier())
])

In [ ]:
model_lr.fit(X_train, y_train)

In [ ]:
model_rf.fit(X_train, y_train)

In [ ]:
model_gb.fit(X_train, y_train)

In [ ]:
def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n{name}")
    print(classification_report(y_test, y_pred))

    roc = roc_auc_score(y_test, y_prob)
    print("ROC-AUC:", roc)

    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"{name} Confusion Matrix")
    plt.show()

    plt.savefig(f"charts/{name}_Confusion_Matrix.png")

    return roc

In [ ]:
roc_lr = evaluate_model(model_lr, X_test, y_test, "Logistic Regression")
roc_rf = evaluate_model(model_rf, X_test, y_test, "Random Forest")
roc_gb = evaluate_model(model_gb, X_test, y_test, "Gradient Boosting")

In [ ]:
def plot_roc(model, name):
    y_prob = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=name)

plt.figure()

plot_roc(model_lr, "LR")
plot_roc(model_rf, "RF")
plot_roc(model_gb, "GB")

plt.plot([0,1],[0,1],'--')
plt.legend()
plt.title("ROC Curve Comparison")
plt.show()
plt.savefig("charts/ROC_Curve_Comparison.png")  

In [ ]:
feature_names = model_rf.named_steps['prep'].get_feature_names_out()
importances = model_rf.named_steps['clf'].feature_importances_

feat_imp = pd.Series(importances, index=feature_names)
top10 = feat_imp.sort_values(ascending=False).head(10)

top10.plot(kind='barh', title="Top 10 Feature Importances")
plt.show()
plt.savefig("charts/Top_10_Feature_Importances.png")